# The Agentic Project Loop — from scratch

> Lesson: [The Agentic Project Loop](https://ml-viz-ruby.vercel.app/wiki/agentic-project-loop)
> · Copy to Drive to run and edit.

We build a **documentation Q&A agent** with **no LLM and no framework** — the "model"
is a small deterministic policy so the whole thing runs offline and identically every
time. That lets you see the *machinery* of every slot without an API key:

`task → context & tools → orchestration → evaluation → guardrails → operations`

The point isn't a smart model; it's the **loop around** the model.

## 1 · Task definition

Goal: answer a question from a docs folder and **cite the doc**. Success = the answer
is grounded in the cited doc, and the agent **refuses** when the docs don't cover the
question. We encode success as checkable functions up front — that *is* the task.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Task:
    question: str
    expected_doc: str | None   # None => the agent SHOULD refuse
    expected_substr: str | None

def is_success(task, answer, cited_doc):
    if task.expected_doc is None:                 # refusal task
        return cited_doc is None and "don't" in answer.lower()
    return cited_doc == task.expected_doc and (task.expected_substr or "") in answer

## 2 · Context & tools

The docs folder, and the two tools the agent may call. `search` returns candidate doc
ids by keyword overlap; `read` returns a doc's text. This is the agentic analogue of
the ML *data* stage — it decides what raw material reaches the decision.

In [ ]:
DOCS = {
    "billing":  "Refunds are issued within 5 business days to the original payment method.",
    "shipping": "Standard shipping takes 3-5 days. Express shipping is delivered next day.",
    "returns":  "Items can be returned within 30 days if unused and in original packaging.",
    "account":  "You can reset your password from the login page via 'Forgot password'.",
}

STOPWORDS = {"the","a","an","is","are","of","to","do","how","i","what","you","can",
             "in","on","it","my","your","from","for","with","and","long","take"}

def keywords(text):
    return {w.strip("?.,'") for w in text.lower().split()} - STOPWORDS

def tool_search(query):
    q = keywords(query)
    scored = [(len(q & keywords(text)), doc_id) for doc_id, text in DOCS.items()]
    scored.sort(reverse=True)
    return [doc_id for score, doc_id in scored if score > 0][:3]   # top-3 non-zero hits

def tool_read(doc_id):
    return DOCS.get(doc_id, "")

TOOLS = {"search": tool_search, "read": tool_read}

### The "model" — a deterministic policy

A real agent would put the question + tool results into a prompt and let an LLM decide
the next action. We stub that with a rule-based policy that emits the **same kind of
structured tool calls** an LLM would. It plans: `search` → `read` the top hit →
`answer`.

In [ ]:
def policy(question, memory):
    # memory is the list of (action, args, observation) so far.
    if not memory:
        return ("search", question)
    last_action = memory[-1][0]
    if last_action == "search":
        hits = memory[-1][2]
        if not hits:                       # nothing retrieved -> refuse (guardrail lever)
            return ("answer", (None, "Sorry, I don't have docs covering that."))
        return ("read", hits[0])           # read the top candidate
    if last_action == "read":
        doc_id, text = memory[-1][1], memory[-1][2]
        return ("answer", (doc_id, text))  # cite the doc we read
    return ("answer", (None, "Sorry, I don't know."))

## 3 · Orchestration loop

The **plan → act → observe** cycle. The policy proposes an action; the runtime executes
tools and feeds the observation back; the loop repeats until an `answer`. Crucially it
is **bounded** by `max_steps` — an unbounded agent loop is a production incident.

In [ ]:
def run_agent(question, max_steps=5, tool_allowlist=("search", "read")):
    memory, trace = [], []
    for step in range(max_steps):
        action, args = policy(question, memory)
        trace.append((step, action, args if action != "answer" else args[0]))
        if action == "answer":
            cited_doc, text = args
            return text, cited_doc, trace
        if action not in tool_allowlist:               # guardrail: blocked tool
            memory.append((action, args, "BLOCKED"))
            continue
        obs = TOOLS[action](args)
        memory.append((action, args, obs))
    return "Sorry, I ran out of steps.", None, trace   # bounded-loop safety net

ans, doc, trace = run_agent("How long do refunds take?")
print("answer:", ans)
print("cited:", doc)
for s in trace: print("  step", s)

**What to notice.** One question drove a two-tool trajectory (`search` → `read` →
`answer`) that ends grounded in a specific doc. That trajectory — not just the final
string — is what stage 4 has to judge.

## 4 · Evaluation — outcome *and* trajectory

Score a small eval set on both axes: **outcome** (right answer, right citation) and
**trajectory** (did it use the tools sensibly, without looping or wasting steps?).

In [ ]:
EVAL = [
    Task("How long do refunds take?", "billing", "5 business days"),
    Task("How do I reset my password?", "account", "Forgot password"),
    Task("What is the capital of France?", None, None),   # not in docs -> must refuse
]

def evaluate(eval_set):
    outcomes, steps_used = [], []
    for task in eval_set:
        ans, doc, trace = run_agent(task.question)
        outcomes.append(is_success(task, ans, doc))
        steps_used.append(len(trace))
    return outcomes, steps_used

outcomes, steps_used = evaluate(EVAL)
print(f"outcome success: {sum(outcomes)}/{len(outcomes)}")
print(f"avg trajectory length: {sum(steps_used)/len(steps_used):.1f} steps")
assert all(outcomes), "an eval case failed — the agent is not doing its task"
print("all eval cases pass (outcome); trajectories are short and tool-grounded")

## 5 · Guardrails — contain what eval can't prevent at runtime

Two levers already in the loop: **refuse on empty retrieval** (no docs -> don't
hallucinate) and a **tool allow-list**. Here's the allow-list doing its job when a
compromised policy tries to call a destructive tool — the model's *choice* was wrong,
but the guardrail bounds the blast radius.

In [ ]:
def rogue_policy(question, memory):
    return ("delete_all", None)   # a prompt-injected / buggy policy tries something nasty

# Temporarily swap the policy; delete_all is NOT on the allow-list.
import types
_saved = policy
def run_with(policy_fn, question):
    globals()['policy'] = policy_fn
    try:    return run_agent(question, tool_allowlist=("search", "read"))
    finally: globals()['policy'] = _saved

ans, doc, trace = run_with(rogue_policy, "How long do refunds take?")
assert "delete_all" not in TOOLS, "delete_all must not even exist as a tool"
blocked = any(a == "delete_all" for _, a, _ in trace)
print("rogue tool attempted:", blocked, "| executed: False (blocked by allow-list)")
print("final answer:", ans)

**What to notice.** `delete_all` was *proposed* every step but *never executed* —
the guardrail sits between the model's choice and the world. Evaluation measures
behaviour; guardrails bound it. You need both.

## 6 · Operations feedback

In production you trace every run — cost, latency, tool-error rate, failed trajectories
— version the prompts so changes are diffable, and **sample live traces back into the
eval set**. That feedback redefines the task (new refusal cases, new tools). Here's the
skeleton of a trace record; in real life it goes to an observability backend.

In [ ]:
def traced_run(question):
    ans, doc, trace = run_agent(question)
    return {
        "question": question, "answer": ans, "cited_doc": doc,
        "n_steps": len(trace), "tools_called": [a for _, a, _ in trace if a in TOOLS],
        "refused": doc is None,
    }

for q in ["How long do refunds take?", "What is the capital of France?"]:
    print(traced_run(q))

## ✏️ Your turn — the agentic slot-placement drill

For each technique, name the single slot it primarily changes:

`task`, `context`, `orchestration`, `evaluation`, `guardrails`, `operations`.

In [ ]:
# TODO(you): map each technique to the slot it primarily changes.
placements = {
    "reranking retrieved chunks":     "...",   # reorder search hits before the model sees them
    "self-reflection / critique step":"...",   # the agent reviews its own draft before answering
    "output schema validation":       "...",   # reject answers that don't match a required JSON shape
    "LLM-as-judge eval set":          "...",   # score trajectories offline with another model
    "prompt version pinning":         "...",   # freeze & diff the prompt used in production
    "defining a refusal policy":      "...",   # decide up front when the agent must decline
}

In [ ]:
solution = {
    "reranking retrieved chunks":      "context",
    "self-reflection / critique step": "orchestration",
    "output schema validation":        "guardrails",
    "LLM-as-judge eval set":           "evaluation",
    "prompt version pinning":          "operations",
    "defining a refusal policy":       "task",
}
for k, v in placements.items():
    assert v == solution[k], f"{k!r}: which slot was breaking?"
print("all placements correct — you can read agent techniques as slot-changes now.")

<details>
<summary>Solution & reasoning</summary>

| Technique | Slot | Why |
|---|---|---|
| reranking retrieved chunks | `context` | it changes *what the model sees*, before any reasoning |
| self-reflection / critique | `orchestration` | an extra step in the runtime loop |
| output schema validation | `guardrails` | it bounds what may leave the system |
| LLM-as-judge eval set | `evaluation` | offline scoring of outcome/trajectory |
| prompt version pinning | `operations` | production change management |
| defining a refusal policy | `task` | it's part of *what success means*, set before building |

</details>

## Key takeaways

- The agent loop is machinery around a model you don't retrain — you move the
  **context, tools, and loop**, not the weights.
- **Evaluation and guardrails are different jobs**: eval *measures* behaviour offline;
  guardrails *bound* it at runtime. The rogue-tool demo needed the allow-list, not a
  better eval.
- The loop closes through **operations**: traces sampled back into the eval set
  redefine the task — exactly the shape of the [ML project loop](https://ml-viz-ruby.vercel.app/wiki/ml-project-loop),
  with a frozen model in the middle.